## Loading the data
Load raw posts as .csv from Azure blob storage.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
import pandas as pd
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_KEY") # Your API key here

gold_label = 'supercategory'

dataset_name = "wikitext-103-hoyle-2022" # "bills-hoyle-2022" or "wikitext-103-hoyle-2022"
test_data = pd.read_json(f"../data/{dataset_name}/test.metadata.jsonl", lines=True)

In [ ]:
test_data.head()

In [ ]:
%matplotlib inline

In [ ]:
import logging
for handler in logging.root.handlers[:]:
   logging.root.removeHandler(handler)
# logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.NOTSET,
                   filename='../data/logs/lda-logs.log')

## Apply pre-processing
Note, first the contents are still full text and the mean length is in chars, after preprocessing it into bag of words it is a list of strings and length is in word count therefor

In [ ]:
input_column = "tokenized_text"

test_docs = test_data[input_column].tolist()
test_docs = [doc.split() for doc in test_docs]
print(len(test_docs))
print(test_docs[0][:200])

In [ ]:
# Remove rare and common tokens.
from gensim.corpora import Dictionary

# Create a dictionary representation of the documents.
dictionary = Dictionary(test_docs)

# Bag-of-words representation of the documents.
test_corpus = [dictionary.doc2bow(doc) for doc in test_docs]

print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(test_corpus))

In [ ]:
#test_doc_bow = dictionary.doc2bow(test_doc)
#test_doc_bow

# Training the model: LDA

In [ ]:
run = "paper-bm-lda-v1-wiki-final-gibbs"

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:8080")

In [ ]:
def get_or_create_experiment(experiment_name):
    """
    Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.

    This function checks if an experiment with the given name exists within MLflow.
    If it does, the function returns its ID. If not, it creates a new experiment
    with the provided name and returns its ID.

    Parameters:
    - experiment_name (str): Name of the MLflow experiment.

    Returns:
    - str: ID of the existing or newly created MLflow experiment.
    """

    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

In [ ]:
llmbt_experiment_id = get_or_create_experiment("ICNLSP Experiment Gibbs LDA")
llmbt_experiment_id

In [ ]:
import numpy as np

def get_document_vectors(lda_model, corpus, num_topics):
    doc_vectors = []
    
    for bow in corpus:
        topic_distribution = lda_model.get_document_topics(bow, minimum_probability=0)
        # Convert sparse topic distribution into a dense vector
        dense_vector = np.zeros(num_topics)
        for topic_id, prob in topic_distribution:
            dense_vector[topic_id] = prob
        doc_vectors.append(dense_vector)
    
    return np.array(doc_vectors)

In [ ]:
from typing import List
import json
import tomotopy as tp
import shutil

from openai import OpenAI
from wordcloud import WordCloud
from textonomy.utils import metrics
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import os
import mlflow
from mlflow.models import infer_signature
import mlflow.data
os.environ["TOKENIZERS_PARALLELISM"] = "true"

mlflow.openai.autolog()

openai = OpenAI(api_key=openai_api_key)  # Make sure openai_api_key is defined

# Embeddings for internal clustering metric
embedding_model = SentenceTransformer('all-mpnet-base-v2') # or other modernbert model
embeddings = embedding_model.encode(test_docs)

ks = [16, 13, 15]

for i, k in enumerate(ks):

    model_name = "lda_tomotopy"
    printprefix = str(i) + " - " + model_name + ": "
    training_info_tag = "wiki attempt #1 with tomotopy"

    with mlflow.start_run(experiment_id=llmbt_experiment_id, run_name=f"{model_name}_k={k}"):
        mlflow.set_tag("Training Info", training_info_tag)
        mlflow.log_param("model_name", model_name)

        dataset = mlflow.data.from_pandas(test_data, source="Azure Blob Storage", name=dataset_name)
        mlflow.log_input(dataset, context=dataset_name)

        model_params = {
            "k": k,
            "iterations": 200,
            "alpha": 5/k,
            "eta": 0.01
        }
        mlflow.log_params(model_params)

        print(printprefix + "LDA model fitting using tomotopy (Gibbs Sampling)")

        # Create a tomotopy LDA model
        lda_model = tp.LDAModel(k=model_params["k"], alpha=model_params["alpha"], eta=model_params["eta"], min_cf=5)

        # Add documents to the model
        for doc in test_docs:
            lda_model.add_doc(doc)

        # Train the model
        lda_model.train(model_params["iterations"])

        print(printprefix + "LDA model fitting done")
        print(printprefix + "Calculating topics for test data")

        # Infer topics for test documents
        test_docs_for_infer = [lda_model.make_doc(doc) for doc in test_docs]
        test_doc_topics, _ = lda_model.infer(test_docs_for_infer)
        
        # Get predicted labels - test_doc_topics is a numpy array of shape (n_docs, k)
        # where each row contains topic probabilities for a document
        predicted_labels = np.argmax(test_doc_topics, axis=1).tolist()
        test_data['predicted_label'] = predicted_labels

        # Log clustered test data to mlflow
        clustered_test_data = "../data/tmp/clustered_test_data.json"
        test_data[['id', gold_label, 'predicted_label']].to_json(clustered_test_data, orient="records")
        mlflow.log_artifact(local_path=clustered_test_data, artifact_path="clustered_test_data.json")
        os.remove(clustered_test_data)

        topics = test_data['predicted_label'].to_list()

        print(printprefix + "Calculating metrics")
        
        # The metrics evaluation function might need to be adapted for the new topic format
        # For now, we create a mock lda_model object that has a `show_topics` method
        class MockLdaModel:
            def __init__(self, tomotopy_model):
                self._model = tomotopy_model
            
            def show_topics(self, num_topics=-1, num_words=10, formatted=True):
                if num_topics == -1:
                    num_topics = self._model.k
                
                topics_to_show = []
                for i in range(num_topics):
                    words = self._model.get_topic_words(i, top_n=num_words)
                    if formatted:
                        topics_to_show.append((i, " + ".join([f'"{w[0]}"' for w in words])))
                    else:
                        topics_to_show.append((i, [(w[0], w[1]) for w in words]))
                return topics_to_show

            def show_topic(self, topicid, topn=10):
                return [(w[0], w[1]) for w in self._model.get_topic_words(topicid, top_n=topn)]

        mock_model = MockLdaModel(lda_model)

        purpose = "A taxonomy to organise these articles into main categories. Aim at around 10-20 categories. Examples: 'Music', 'Social sciences and society'..."
        topk = 10

        metric_dict, llm_usefulness_reasoning = metrics.evaluate_lda_model(lda_model=mock_model, purpose=purpose, openai_client=openai, topk=topk, pp_ref_docs=test_docs, model_labels=topics, gold_labels=test_data[gold_label].to_list(), embeddings=embeddings)
        
        mlflow.log_param("eval_topk", topk)
        metric_dict['outlier_ratio'] = 0
        metric_dict['num_test_docs'] = test_data.id.nunique()
        metric_dict['num_topics'] = model_params["k"]
        
        valid_metrics, invalid_metrics = {}, {}
        for k_met, v in metric_dict.items():
            try:
                valid_metrics[k_met] = float(v)
            except (ValueError, TypeError):
                invalid_metrics[k_met] = v
        mlflow.log_metrics(valid_metrics)
        if invalid_metrics:
            invalid_metrics_path = "../data/tmp/invalid_metrics.txt"
            with open(invalid_metrics_path, "w") as f:
                json.dump(invalid_metrics, f, indent=4)
            mlflow.log_artifact(invalid_metrics_path)
            os.remove(invalid_metrics_path)
        
        mlflow.log_text(purpose, "user_defined_purpose_for_U_LLM_USEFULNESS.txt")
        mlflow.log_text(str(llm_usefulness_reasoning), "U_LLM_USEFULNESS_reasoning.txt")

        metric_df = pd.DataFrame.from_dict(metric_dict, orient="index", columns=["value"])
        display(metric_df)

        # Save and log topic info
        topic_info_path = "../data/tmp/tomotopy_topic_info.txt"
        with open(topic_info_path, "w") as f:
            for i in range(lda_model.k):
                f.write(f"Topic #{i}\n")
                for word, prob in lda_model.get_topic_words(i, top_n=20):
                    f.write(f"\t{word}: {prob}\n")
        mlflow.log_artifact(topic_info_path, artifact_path="lda_topic_info_data")
        os.remove(topic_info_path)

        # topic_words = mock_model.show_topic(0, topn=20)
        # wordcloud = WordCloud(width=800, height=400).generate_from_frequencies(dict(topic_words))
        # wordcloud_file = "../data/tmp/wordcloud.png"
        # wordcloud.to_file(wordcloud_file)
        # mlflow.log_artifact(wordcloud_file)
        
        print("Logged run '" + str(run) + "' to MLflow.")
        print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
import os
from openai import OpenAI
from textonomy.models import Textonomy
os.environ["TOKENIZERS_PARALLELISM"] = "true"

openai_api_key = os.getenv("OPENAI_KEY") # Your API key here
client = OpenAI(api_key=openai_api_key)

model = Textonomy()

# Calculate LDA Stability Baseline

In [ ]:
import itertools
from typing import Dict

artifact_path = "clustered_test_data.json/clustered_test_data.json"

run_ids = [
    "18c174adc4e6417da79958725d738258",
    "4c74e6fcf3c44280bd9e9dc06427e4c3",
    "24be78b4967548f3bf487446d698ead0",
    "5fc86fab2380486cbbabbe3162dbc18e",
    "3d6e37eb12a34787aa03d873c57453c3",
    "536cdcaa398a43debaf57f35258279b5",
    "137919f5145c41d4b7c9197411e52832",
    "d0cfc32aaaf94483aef38703306914a0",
    "23bd60d0aa1c401780d7db6f582400cf",
    "a9fcd6319b6a4bc98800b04fd8d300ed"
]

artifact_uris = []
for run_id in run_ids:
    try:
        artifact_uri = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)
        artifact_uris.append(artifact_uri)
    except Exception as e:
        print(f"Error downloading artifact for run_id {run_id}: {e}")
        # Handle the error appropriately, e.g., skip this run or raise the exception
        continue  # Skip to the next run ID


dataframes = []
for artifact_uri in artifact_uris:
    try:
        with open(artifact_uri, "r") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        dataframes.append(df)
    except Exception as e:
            print(f"Error loading or processing artifact from {artifact_uri}: {e}")
            continue

if len(dataframes) < 2:
    raise ValueError(f"Fewer than 2 useable artifacts, could not create comparison pairs")

# Initialize a dictionary to store the sum of each metric and a counter
metric_sums: Dict[str, float] = {}
metric_counts: Dict[str, int] = {}
metric_dicts = []

# Iterate over all pairs of runs
for df1, df2 in itertools.combinations(dataframes, 2):
    try:
        # Calculate external cluster metrics
        metrics = model.calculate_external_cluster_metrics(y_true=df1['predicted_label'], y_pred=df2['predicted_label'])
        metric_dicts.append(metrics)

        # Accumulate the metrics
        for metric_name, metric_value in metrics.items():
            metric_sums[metric_name] = metric_sums.get(metric_name, 0) + metric_value
            metric_counts[metric_name] = metric_counts.get(metric_name, 0) + 1
    except Exception as e:
            print(f"Error calculating or processing metrics: {e}")
            continue

# Calculate average metrics
average_metrics = {}
for metric_name in metric_sums:
    average_metrics[metric_name] = metric_sums[metric_name] / metric_counts[metric_name]

print("Average metrics:")
for metric_name, metric_value in average_metrics.items():
    print(f"{metric_name}: {metric_value}")


In [ ]:
metric_dicts

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def purity(model_labels, gold_labels): 
    """
    Calculates the Purity metric as described in https://aclanthology.org/P16-1110/
    "ALTO: Active Learning with Topic Overviews for Speeding Label Induction and Document Labeling"
    
    For sanity check - The purity of any two user labels should be 1
    """
    assert len(model_labels) == len(gold_labels)

    # somewhat faster than a pure-python implementation
    purity_sum = (
        pd.DataFrame({"pred": model_labels, "true": gold_labels})
          .groupby(["pred", "true"], as_index=False)
          .size()
          .groupby("pred")["size"]
          .max()
          .sum()
    )
    return purity_sum / len(model_labels)

# Ground truth and predicted labels
y_true = test_data[gold_label].to_list()
y_pred = topics

# Compute metrics
ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred)
purity1 = purity(model_labels=y_pred, gold_labels=y_true)

# Print results
print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
print(f"Purity: {purity1:.4f}")


In [ ]:
import umap
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-mpnet-base-v2') # or other modernbert model
embeddings = embedding_model.encode(test_data["text"].tolist())
# For full embeddings:
#embeddings = load_embeddings_numpy(filepath="data/wikitext-103-hoyle-2022/train_embeddings.npy")

# 2. UMAP to 2D
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2 = reducer.fit_transform(embeddings)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder

def load_embeddings_numpy(filepath="embeddings.npy"):
    """Loads embeddings from a NumPy array."""
    return np.load(filepath)

def evaluate_text_embeddings(labels, embeddings):
    """
    Calculates internal cluster metrics, and prints the results.

    Args:
        train_data (pd.DataFrame): DataFrame with "text" and "category" columns.
    """

    # 1. Embeddings with modernbert
    #model = SentenceTransformer('all-mpnet-base-v2')
    #embeddings = model.encode(train_data["text"].tolist())

    # 2. Encode labels for metric calculation
    le = LabelEncoder()
    labels = le.fit_transform(labels)

    # 3. Calculate internal cluster metrics
    silhouette = silhouette_score(embeddings, labels, metric="cosine")
    davies_bouldin = davies_bouldin_score(embeddings, labels)
    calinski_harabasz = calinski_harabasz_score(embeddings, labels)

    # 4. Print results
    print(f"Silhouette Score: {silhouette:.4f}")
    print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")
    print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")

# Example usage (replace with your actual data)
#data_tes = {'text': ["This is a happy sentence.", "Another happy one.", "A sad story.", "Very sad indeed.", "Neutral text.", "A typical neutral example."],
#        'category': ["positive", "positive", "negative", "negative", "neutral", "neutral"]}
#train_data_tes = pd.DataFrame(data_tes)

print("Gold label clustering on topic prob vectors:")
evaluate_text_embeddings(test_data[gold_label], topic_probs)
print("Most probable topic label clustering on topic prob vectors:")
evaluate_text_embeddings(topics, topic_probs)
print("Most probable topic label clustering on embeddings:")
evaluate_text_embeddings(topics, embeddings=embeddings)
print("Gold label clustering on embeddings:")
evaluate_text_embeddings(test_data[gold_label], embeddings=embeddings)

In [ ]:
model.show_topics(num_topics=-1, num_words=10, formatted=True)